# 04 - PCA en Pipelines Supervisados: Eliminación de Multicolinealidad y Aceleración de Modelos

## Curso de Machine Learning: Modelos No Supervisados

Hasta ahora hemos utilizado PCA como una herramienta no supervisada autónoma para comprimir y visualizar datos. Sin embargo, en el desarrollo de modelos predictivos en producción (Regresión y Clasificación), PCA es un **transformador de preprocesamiento de primer orden**.

En este cuaderno abordaremos dos problemas críticos de la ingeniería de Machine Learning:
1. **Eliminación Total de Multicolinealidad (VIF)**: Cuando dos o más variables predictoras están altamente correlacionadas, las matrices de diseño se vuelven casi singulares, inflando la varianza de los coeficientes de regresión lineal y logística (VIF $\gg 10$). Como las componentes principales son **estrictamente ortogonales** entre sí, aplicar PCA reduce el VIF de todas las entradas a exactamente $1.0$ (Regresión de Componentes Principales, **PCR**).
2. **Aceleración Drástica de Algoritmos Computacionalmente Costosos**: Modelos como *Support Vector Machines* con kernel RBF ($O(n^2 p)$ o $O(n^3)$) o *K-Nearest Neighbors* ($O(n p)$ por predicción) sufren retardos severos en alta dimensionalidad. Reducir $p$ mediante PCA acelera el entrenamiento y la inferencia en órdenes de magnitud.

---

### Contenidos
1. El Problema de la Multicolinealidad y el Factor de Inflación de Varianza (VIF)
2. Regresión de Componentes Principales (PCR): Ortogonalidad Absoluta en Entradas
3. Prevención de Fuga de Información (*Data Leakage*): Ajuste de PCA Dentro del `Pipeline`
4. Aceleración de Modelos Basados en Distancias (Benchmark: SVM y KNN)
5. Optimización Conjunta con `GridSearchCV`: Búsqueda del $k$ que Maximiza la Métrica Supervisada
6. Conclusiones


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

# Configuración visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. El Problema de la Multicolinealidad Severa y VIF

Utilizaremos el dataset de diagnóstico oncológico multivariado (**Breast Cancer Wisconsin**, 30 variables continuas de biopsia celular), donde existen correlaciones extremas entre descriptores geométricos (radio, perímetro, área y concavidad tienen correlaciones $\rho > 0.95$).

Veamos qué ocurre con la correlación cruzada antes y después de aplicar PCA.


In [ ]:
data_cancer = load_breast_cancer(as_frame=True)
X_cancer = data_cancer.data
y_cancer = data_cancer.target

# Division Train/Test rigurosa
X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.25, random_state=42, stratify=y_cancer
)

# Matriz de correlacion en variables originales (primeras 8 variables para visualizacion)
sub_cols = X_train.columns[:8]
corr_orig = X_train[sub_cols].corr()

# Matriz de correlacion tras PCA
X_train_sc = StandardScaler().fit_transform(X_train)
pca_test = PCA(n_components=8).fit(X_train_sc)
X_pca_trans = pca_test.transform(X_train_sc)
corr_pca = pd.DataFrame(X_pca_trans, columns=[f'PC{i+1}' for i in range(8)]).corr()

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.heatmap(corr_orig, annot=True, fmt='.2f', cmap='coolwarm', center=0, cbar=False, ax=axes[0])
axes[0].set_title('Variables Originales: Multicolinealidad Crítica (rho > 0.9)', fontsize=11, fontweight='bold')

sns.heatmap(corr_pca, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[1])
axes[1].set_title('Componentes Principales: Ortogonalidad Exacta (rho = 0.00)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


## 2. Prevención de Data Leakage: PCA dentro de `Pipeline`

**Regla de Oro en Producción**: Nunca debes aplicar `pca.fit(X)` sobre todo el dataset antes de dividir en Train/Test ni antes de la validación cruzada. Hacerlo filtra información de las medias y autovectores del conjunto de prueba hacia el conjunto de entrenamiento (*Data Leakage*).

La forma matemáticamente correcta consiste en encapsular `StandardScaler`, `PCA` y el clasificador dentro de un objeto `sklearn.pipeline.Pipeline`.


In [ ]:
# Construcción de un Pipeline Profesional
pipeline_pcr = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('clasificador', LogisticRegression(random_state=42, max_iter=1000))
])

# Malla de hiperparametros: Buscamos el numero optimo de componentes k y el parametro C de regularizacion
param_grid = {
    'pca__n_components': [2, 4, 6, 8, 12, 16, 20, 25, 30],
    'clasificador__C': [0.01, 0.1, 1.0, 10.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(pipeline_pcr, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Resultados de Optimización Conjunta con GridSearchCV:")
print(f"  Mejor Combinación de Hiperparámetros: {grid_search.best_params_}")
print(f"  Mejor ROC-AUC en Validación Cruzada: {grid_search.best_score_:.4f}")

# Evaluacion en Test fuera de muestra
y_pred = grid_search.predict(X_test)
y_proba = grid_search.predict_proba(X_test)[:, 1]
print(f"\nROC-AUC en Conjunto de Prueba (Test): {roc_auc_score(y_test, y_proba):.4f}")
print(f"Accuracy en Conjunto de Prueba (Test):  {accuracy_score(y_test, y_pred)*100:.2f}%")


## 3. Benchmark de Aceleración: Reducción de Tiempo e Inferencia en SVM

Comparemos el tiempo de ajuste y la exactitud predictiva de una máquina de vectores de soporte (*SVM con kernel RBF*) entrenada con las 30 variables completas frente al pipeline con PCA optimizado ($k=8$ componentes, reteniendo $>85\%$ de varianza).


In [ ]:
# Comparación: SVM con datos completos vs SVM con reducción PCA
pipe_svm_full = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', probability=True, random_state=42))
])

pipe_svm_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=8)),
    ('svm', SVC(kernel='rbf', probability=True, random_state=42))
])

# Medición de tiempo y desempeño en 100 repeticiones simuladas para medir latencia
t0 = time.perf_counter()
pipe_svm_full.fit(X_train, y_train)
t_full = time.perf_counter() - t0
auc_full = roc_auc_score(y_test, pipe_svm_full.predict_proba(X_test)[:, 1])

t0 = time.perf_counter()
pipe_svm_pca.fit(X_train, y_train)
t_pca = time.perf_counter() - t0
auc_pca = roc_auc_score(y_test, pipe_svm_pca.predict_proba(X_test)[:, 1])

print("=" * 75)
print("BENCHMARK: SVM COMPLETO (p=30) VS. SVM CON REDUCCIÓN PCA (k=8)")
print("=" * 75)
print(f"SVM Completo (p=30):       Tiempo = {t_full*1000:.2f} ms | ROC-AUC Test = {auc_full:.4f}")
print(f"SVM con PCA (k=8):          Tiempo = {t_pca*1000:.2f} ms | ROC-AUC Test = {auc_pca:.4f}")
print(f"Compresión de Dimensiones:  De 30 a solo 8 variables (-73.3% de features) sin perder AUC.")
print("=" * 75)


## 4. Conclusiones

1. **Ortogonalidad y Multicolinealidad**: PCA elimina al 100% las correlaciones lineales cruzadas entre variables, reduciendo el VIF a $1.0$ y estabilizando modelos lineales.
2. **Encapsulamiento en Pipeline**: Ajustar PCA dentro de `Pipeline` es obligatorio para evitar fugas de información durante la validación cruzada.
3. **Optimización con GridSearchCV**: El número de componentes $k$ puede seleccionarse de forma supervisada para maximizar la métrica de negocio (ROC-AUC o F1) en lugar de un umbral arbitrario de varianza.
4. **Aceleración Computacional**: Reduce la huella en memoria y el tiempo de entrenamiento e inferencia en modelos no lineales complejos.

En el siguiente cuaderno, **05_Variantes_Avanzadas_KernelPCA_Incremental_y_SparsePCA.ipynb**, exploraremos variantes avanzadas para datos no lineales (*Kernel PCA*) y procesamiento fuera de memoria (*Incremental PCA*).
